In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import pickle
import os

# Load cleaned data
df = pd.read_csv('../data/processed/churn_cleaned.csv')

X = df.drop(columns=['Churn'])
y = df['Churn']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

X shape: (7043, 19)
y shape: (7043,)

Target distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64


In [2]:
# Define numeric and categorical columns
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_cols = X.select_dtypes(include='object').columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

# Build preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

print("\nPreprocessor built successfully")

Numeric columns: ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Preprocessor built successfully


In [3]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("\nTrain churn distribution:")
print(y_train.value_counts(normalize=True).round(3) * 100)

# Fit preprocessor on train, transform both
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("\nProcessed train shape:", X_train_processed.shape)
print("Processing done successfully")

# Save preprocessor
os.makedirs('../models', exist_ok=True)
with open('../models/preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

print("Preprocessor saved to models/preprocessor.pkl")

Train size: (5634, 19)
Test size: (1409, 19)

Train churn distribution:
Churn
0    73.5
1    26.5
Name: proportion, dtype: float64

Processed train shape: (5634, 44)
Processing done successfully
Preprocessor saved to models/preprocessor.pkl


In [4]:
# Save processed splits for training notebook
import numpy as np

np.save('../data/processed/X_train.npy', X_train_processed)
np.save('../data/processed/X_test.npy', X_test_processed)
np.save('../data/processed/y_train.npy', y_train.values)
np.save('../data/processed/y_test.npy', y_test.values)

print("Saved all splits:")
print("  X_train:", X_train_processed.shape)
print("  X_test:", X_test_processed.shape)
print("  y_train:", y_train.shape)
print("  y_test:", y_test.shape)

Saved all splits:
  X_train: (5634, 44)
  X_test: (1409, 44)
  y_train: (5634,)
  y_test: (1409,)
